# Q53: Feature Importance in RF & GBDT
**Topic:** Classical-ml | **Difficulty:** Medium | **Time:** 10 min
**Sheet:** Grind75ML

---

## Question
How does feature importance work in Random Forest / GBDT?

## Detailed Answer

### Mean Decrease in Impurity (MDI) — Default
For each feature, sum the total reduction in impurity (Gini or entropy) across all trees and all splits using that feature.

$$\text{Importance}(f) = \sum_{\text{trees}} \sum_{\text{nodes using } f} n_\text{node} \cdot \Delta \text{impurity}$$

- **Fast** (computed during training)
- **Biased**: Favors high-cardinality features and correlated features
- May not reflect true predictive power

### Permutation Importance — Better
1. Compute baseline score on validation data
2. For each feature: shuffle it across samples, recompute score
3. Importance = decrease in score after permutation

- **Model-agnostic**, works for any model
- **Unbiased** — measures actual predictive contribution
- Slower (need to re-evaluate for each feature)
- Correlated features: importance splits between them

### SHAP Values — Gold Standard
Based on game theory (Shapley values):
- Computes exact contribution of each feature to each prediction
- **TreeSHAP**: Efficient exact computation for tree-based models — $O(TLD^2)$
- Global importance = mean |SHAP value| across samples
- Also gives **local explanations** (per-prediction feature importance)

### Comparison
| Method | Speed | Bias | Local Explainability |
|--------|-------|------|---------------------|
| MDI | Fast | High (cardinality) | No |
| Permutation | Medium | Low | No |
| SHAP | Slow (fast for trees) | None | Yes |

In [ ]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.inspection import permutation_importance

X, y = make_classification(n_samples=1000, n_features=10, n_informative=3, random_state=42)
rf = RandomForestClassifier(n_estimators=100, random_state=42).fit(X, y)

# MDI (default)
print('MDI Importance:', np.round(rf.feature_importances_, 3))

# Permutation importance
perm_imp = permutation_importance(rf, X, y, n_repeats=10, random_state=42)
print('Permutation Importance:', np.round(perm_imp.importances_mean, 3))

print(f'\nTop-3 by MDI: {np.argsort(rf.feature_importances_)[-3:][::-1]}')
print(f'Top-3 by Permutation: {np.argsort(perm_imp.importances_mean)[-3:][::-1]}')

## Key Takeaways
- **Don't trust default (MDI) importance** for high-cardinality features — always validate with permutation
- Use **SHAP** when you need explanations for stakeholders or regulators
- **Permutation importance** is the best balance of reliability and simplicity
- Correlated features dilute importance — combine with correlation analysis